# BlackBox — Phase 5: ADD / UPDATE / DELETE / NOOP

The slide deck's claim (slide 19, "Mem0 builder"):

> Mem0 compares the new fact with similar memories and decides whether to **add**, **update**,
> **delete**, or **ignore** it to keep the memory store clean and non-redundant.

Phase 2 tested exactly one case of this -- the 737-to-787 fleet change -- and found `add()`
only ever reported ADD, and the old fact was never removed from storage. That was one
incidental result from a narrative-driven test. This notebook tests all four branches
**deliberately and separately**, so we can say with actual evidence which of the four claimed
behaviors mem0 really exhibits.

Same `eng_01` data, reused as-is.


## 0. Setup and starting snapshot

In [ ]:
import os
from dotenv import load_dotenv
from mem0 import MemoryClient

load_dotenv()
client = MemoryClient(api_key=os.getenv("MEM0_API_KEY"))

starting = client.get_all(filters={"user_id": "eng_01"})
starting_facts = [item["memory"] for item in starting.get("results", [])]

print(f"Starting count: {len(starting_facts)}")
for f in starting_facts:
    print(" -", f)

## Test 1 — ADD

Expectation: a fact with nothing similar already stored should just get added, no
reconciliation needed. We pick something topically unrelated to hydraulics on purpose, so
there's nothing for mem0 to compare it against.


In [ ]:
before = client.get_all(filters={"user_id": "eng_01"})
before_count = len(before.get("results", []))

add_result = client.add(
    "User is also certified on Airbus A320 avionics, separate from the Boeing hydraulics work.",
    user_id="eng_01",
)
print("add() returned:", add_result)

after = client.get_all(filters={"user_id": "eng_01"})
after_count = len(after.get("results", []))

print(f"\nCount before: {before_count}, after: {after_count}")
print("ADD confirmed" if after_count > before_count else "No new memory was actually stored")

## Test 2 — UPDATE

Expectation: a *correction* to something already stored should edit the existing memory in
place, not just pile on a second, contradicting one. We reference the exact backup-system
fact from earlier phases and issue a correction to it.


In [ ]:
before = client.get_all(filters={"user_id": "eng_01"})
before_facts = {item["memory"] for item in before.get("results", [])}

update_result = client.add(
    "Correction: the Standby Hydraulic System backup is driven by an electric motor only, "
    "not the APU -- that detail I gave earlier was wrong.",
    user_id="eng_01",
)
print("add() returned:", update_result)

after = client.get_all(filters={"user_id": "eng_01"})
after_facts = {item["memory"] for item in after.get("results", [])}

removed = before_facts - after_facts
added = after_facts - before_facts

print("\nFacts removed (should be the old, now-wrong version, if UPDATE really edits in place):")
for f in removed:
    print(" -", f)

print("\nFacts added:")
for f in added:
    print(" -", f)

print(f"\nCount before: {len(before_facts)}, after: {len(after_facts)}")
print("Looks like an in-place UPDATE" if removed and added and len(after_facts) == len(before_facts)
      else "Did NOT behave like a clean in-place update -- see the raw before/after above")

## Test 3 — DELETE

This is the exact case from Phase 2, isolated on its own this time. Expectation per the
slide: a directly contradicting fact should cause the old one to be **removed**.

We already have real evidence from Phase 2 that this doesn't happen with MemoryClient v3 --
running it again here, in isolation, is what turns that from an anecdote into a controlled
result.


In [ ]:
before = client.get_all(filters={"user_id": "eng_01"})
before_facts = {item["memory"] for item in before.get("results", [])}
before_count = len(before_facts)

delete_result = client.add(
    "Correction: I do not work on Boeing 737 hydraulics at all anymore -- that information "
    "is now completely out of date and should be disregarded.",
    user_id="eng_01",
)
print("add() returned:", delete_result)

after = client.get_all(filters={"user_id": "eng_01"})
after_facts = {item["memory"] for item in after.get("results", [])}
after_count = len(after_facts)

removed = before_facts - after_facts

print(f"\nCount before: {before_count}, after: {after_count}")
print("\nFacts actually removed:")
if removed:
    for f in removed:
        print(" -", f)
else:
    print(" (none -- every prior 737 fact is still present)")

print("\nDELETE confirmed" if removed else "DELETE did NOT happen -- consistent with Phase 2's finding")

## Test 4 — NOOP

Expectation: restating something already known, in different words, shouldn't grow the
memory count at all.


In [ ]:
before = client.get_all(filters={"user_id": "eng_01"})
before_count = len(before.get("results", []))

noop_result = client.add(
    "Just to confirm again, I do maintenance and engineering work on hydraulic systems.",
    user_id="eng_01",
)
print("add() returned:", noop_result)

after = client.get_all(filters={"user_id": "eng_01"})
after_count = len(after.get("results", []))

print(f"\nCount before: {before_count}, after: {after_count}")
print("NOOP confirmed -- count unchanged" if after_count == before_count
      else f"Count changed by {after_count - before_count} -- NOOP did not hold")

## Summary table

Fill this in with whatever your four cells above actually printed -- don't assume the slide's
description before checking your own output.

| Case | Slide's claim | What we actually observed |
|---|---|---|
| ADD | New, unrelated fact gets stored | *(fill in from Test 1)* |
| UPDATE | Correction edits the existing memory in place | *(fill in from Test 2)* |
| DELETE | Direct contradiction removes the old fact | *(fill in from Test 3 -- Phase 2 already suggests this is "no")* |
| NOOP | Restating known info changes nothing | *(fill in from Test 4)* |

If ADD and NOOP hold up but UPDATE and DELETE don't behave as cleanly as the slide describes,
that's a legitimate, presentable finding -- **"the write path leans on ADD far more than the
architecture diagram suggests, and cleanup is left to the read path instead"** -- which is
exactly the thesis Phases 3 and 4 already built toward. This notebook is what turns that from
a hunch into something you tested four separate times on purpose.
